# LeetCode 4024 — Nearest Available Drone bằng Python

Link đề: https://leetcode.com/problems/nearest-available-drone/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Geometry, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def nearestDrone(self, drones: List[List[int]], target: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi drone `[x, y, range]`; drone **tiếp cận được** mục tiêu nếu **khoảng cách Manhattan** `<= range`; trả về **chỉ số** drone tiếp cận được có khoảng cách **nhỏ nhất** (hòa -> chỉ số nhỏ nhất), hoặc `-1`.
2. Biết tính khoảng cách Manhattan `|x1 - x2| + |y1 - y2|` và phân biệt với khoảng cách Euclid.
3. Nắm mẫu **"lọc điều kiện rồi chọn nhỏ nhất, hòa thì chỉ số nhỏ"**: dùng `<` (chặt) khi cập nhật để giữ chỉ số đầu tiên, hoặc dùng `min` với khóa `(khoảng_cách, chỉ_số)`.
4. Viết 4 cách (vòng lặp, `min` theo khóa, lọc rồi `min`, sắp xếp) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: quên kiểm tra `range`, dùng `<` thay `<=` cho phạm vi, dùng `<=` khi cập nhật làm sai tie-break, nhầm Euclid/Manhattan, trả khoảng cách thay vì chỉ số.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `drones` với `drones[i] = [xi, yi, rangei]` là tọa độ và **tầm bay** của drone `i`, và `target = [tx, ty]` là tọa độ mục tiêu. Drone `i` **tiếp cận được** mục tiêu nếu **khoảng cách Manhattan** `|xi - tx| + |yi - ty|` **nhỏ hơn hoặc bằng** `rangei`.

Trả về **chỉ số** của drone tiếp cận được có khoảng cách Manhattan tới mục tiêu **nhỏ nhất**. Nếu nhiều drone cùng khoảng cách nhỏ nhất, trả về **chỉ số nhỏ nhất**. Nếu không drone nào tiếp cận được, trả về `-1`.

| Ví dụ | `drones`, `target` | Output | Giải thích |
|---|---|---|---|
| 1 | `[[0,0,8],[2,2,9]]`, `[3,4]` | `1` | Drone 0 cách `7 <= 8` (được); drone 1 cách `3 <= 9` (được). Gần nhất là drone 1 |
| 2 | `[[2,1,5],[4,4,5],[6,6,8]]`, `[5,5]` | `1` | Drone 0 cách `7 > 5` (loại). Drone 1 và 2 cùng cách `2`; hòa chọn chỉ số nhỏ -> `1` |
| 3 | `[[4,4,5]]`, `[8,6]` | `-1` | Drone 0 cách `6 > 5`, không tiếp cận được |

Constraints: `1 <= len(drones) <= 100`; `drones[i] = [xi, yi, rangei]`; `target = [tx, ty]`; `-25 <= xi, yi, tx, ty <= 25`; `1 <= rangei <= 100`.

In [1]:
drones, target = [[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]
for i, (x, y, r) in enumerate(drones):
    d = abs(x - target[0]) + abs(y - target[1])
    print(f"drone {i}: khoảng cách={d}, tầm={r}, tiếp cận được? {d <= r}")

drone 0: khoảng cách=7, tầm=5, tiếp cận được? False
drone 1: khoảng cách=2, tầm=5, tiếp cận được? True
drone 2: khoảng cách=2, tầm=8, tiếp cận được? True


## 3. Kiến thức nền

### 3.1. Khoảng cách Manhattan

Khoảng cách Manhattan giữa `(x1, y1)` và `(x2, y2)` là `|x1 - x2| + |y1 - y2|` (đi theo lưới ô vuông, như đi trên các con phố). Khác Euclid `sqrt(dx^2 + dy^2)` (đường chim bay).

In [2]:
x1, y1, x2, y2 = 0, 0, 3, 4
print(abs(x1 - x2) + abs(y1 - y2), ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5)

7 5.0


### 3.2. Giải nén (unpacking) mỗi hàng

`for i, (x, y, r) in enumerate(drones)` lấy cả chỉ số `i` và ba giá trị của mỗi drone. Dễ đọc hơn `drones[i][0]`, `drones[i][1]`, `drones[i][2]`.

In [3]:
for i, (x, y, r) in enumerate([[0, 0, 8], [2, 2, 9]]):
    print(i, x, y, r)

0 0 0 8
1 2 2 9


### 3.3. Tìm nhỏ nhất với tie-break: dùng `<` (chặt)

Duyệt theo chỉ số tăng dần và chỉ cập nhật khi tìm thấy khoảng cách **nhỏ hơn hẳn** (`<`): khi hòa, ta giữ ứng viên **đầu tiên** (chỉ số nhỏ hơn). Dùng `<=` sẽ chuyển sang ứng viên **sau** (sai tie-break).

In [4]:
vals = [(5, 'a'), (3, 'b'), (3, 'c')]
best = None
for d, name in vals:
    if best is None or d < best[0]:
        best = (d, name)
print(best)             # (3, 'b'): giữ b, không đổi sang c

(3, 'b')


### 3.4. `min` với `key` và so sánh tuple

`min(iterable, key=...)` trả phần tử có khóa nhỏ nhất; khi **hòa khóa** thì trả phần tử **đầu tiên** gặp. Tuple so sánh từng phần tử: `(d, i) < (d2, i2)` so `d` trước rồi `i`. Vì vậy `min` trên các tuple `(khoảng_cách, chỉ_số)` tự xử lý tie-break.

In [5]:
print(min([(3, 2), (3, 1), (5, 0)]), min([("b", 3), ("a", 9)]))
print(min([1, 2, 3, 0, 9], key=lambda v: -v))

(3, 1) ('a', 9)
9


### 3.5. `min(..., default=...)` và lọc trước khi `min`

`min` của iterable **rỗng** gây `ValueError`; dùng `default=` để trả giá trị mặc định (ví dụ `(inf, -1)`): `min(generator, default=None)`.

In [6]:
try:
    min([])
except ValueError as e:
    print("ValueError:", e)
print(min([], default=None), min((v for v in [4, 2] if v > 3), default=-1))

ValueError: min() iterable argument is empty
None 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hai câu hỏi cho mỗi drone: (1) **có tiếp cận được không** (`d <= range`)? (2) nếu có, nó **gần cỡ nào**? Ta chỉ **so sánh** các drone tiếp cận được, chọn khoảng cách nhỏ nhất; hòa thì chỉ số nhỏ.

Tính tay ví dụ 2 (`target = (5, 5)`):

| drone | `(x, y, r)` | khoảng cách | `<= r`? | ghi chú |
|---|---|---|---|---|
| 0 | `(2,1,5)` | `3 + 4 = 7` | `7 <= 5` không | loại |
| 1 | `(4,4,5)` | `1 + 1 = 2` | có | ứng viên, tốt nhất = 2 |
| 2 | `(6,6,8)` | `1 + 1 = 2` | có | hòa 2, giữ drone 1 |

Đáp án `1`. Ví dụ 3: `(4,4,5)` cách `(8,6)` là `4 + 2 = 6 > 5` -> `-1`.

In [7]:
def reach_table(drones, target):
    rows = []
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        rows.append((i, d, r, d <= r))
    return rows

for row in reach_table([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]):
    print(row)
print(reach_table([[4, 4, 5]], [8, 6]))

(0, 7, 5, False)
(1, 2, 5, True)
(2, 2, 8, True)
[(0, 6, 5, False)]


## 5. Cách 1 — Brute Force / baseline: duyệt và theo dõi tốt nhất

1. `best_d = vô cực`, `best_i = -1`.
2. Với mỗi drone `i`: `d = |x - tx| + |y - ty|`. Nếu `d <= range` **và** `d < best_d` thì cập nhật `best_d, best_i`.
3. Trả `best_i`.

Điều kiện `d < best_d` (chặt) giữ chỉ số nhỏ khi hòa. Đây gần như dịch nguyên đề, `O(n)`; không có nút thắt thực sự vì phải xét từng drone.

In [8]:
class SolutionLoop:
    def nearestDrone(self, drones, target):
        tx, ty = target
        best_d, best_i = float("inf"), -1
        for i, (x, y, r) in enumerate(drones):
            d = abs(x - tx) + abs(y - ty)
            if d <= r and d < best_d:
                best_d, best_i = d, i
        return best_i

print(SolutionLoop().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]),
      SolutionLoop().nearestDrone([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]),
      SolutionLoop().nearestDrone([[4, 4, 5]], [8, 6]))

1 1 -1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`, `n = len(drones)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách thành **"lọc" + "chọn nhỏ nhất theo khóa `(khoảng_cách, chỉ_số)`"**. Khi so sánh theo cặp `(d, i)`, drone có `d` nhỏ hơn thắng; nếu `d` bằng nhau thì `i` nhỏ hơn thắng: **tie-break nằm ngay trong khóa**, không cần nhớ viết `<` hay `<=`.

`ans = min((d, i) for i ... if d <= range)[1]`, hoặc `-1` nếu không có ứng viên.

**Quy tắc "nếu... thì...":**

- nếu `d > range` thì bỏ qua drone `i`;
- nếu có nhiều drone cùng `d` nhỏ nhất thì chọn `i` nhỏ nhất (tuple tự so sánh);
- nếu không còn ứng viên thì `-1`.

**Invariant:** tại mọi thời điểm, `(best_d, best_i)` là khóa nhỏ nhất trong các drone **đã xét và tiếp cận được**.

## 7. Các cách tiếp theo

### Cách 2 — `min` trên tuple `(d, i)` với `default`

Dựng generator các `(d, i)` của drone tiếp cận được; `min(..., default=(0, -1))` rồi lấy `[1]`.

In [9]:
class SolutionMin:
    def nearestDrone(self, drones, target):
        tx, ty = target
        cands = ((abs(x - tx) + abs(y - ty), i) for i, (x, y, r) in enumerate(drones)
                 if abs(x - tx) + abs(y - ty) <= r)
        return min(cands, default=(0, -1))[1]

print(SolutionMin().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]),
      SolutionMin().nearestDrone([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]),
      SolutionMin().nearestDrone([[4, 4, 5]], [8, 6]))

1 1 -1


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (generator).

### Cách 3 — Lọc thành list rồi chọn

Dựng list các `(d, i)` hợp lệ, nếu rỗng `-1`, còn không `min(list)[1]`. Dễ in ra để gỡ lỗi.

In [10]:
class SolutionFilter:
    def nearestDrone(self, drones, target):
        tx, ty = target
        good = []
        for i, (x, y, r) in enumerate(drones):
            d = abs(x - tx) + abs(y - ty)
            if d <= r:
                good.append((d, i))
        return min(good)[1] if good else -1

print(SolutionFilter().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]),
      SolutionFilter().nearestDrone([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]),
      SolutionFilter().nearestDrone([[4, 4, 5]], [8, 6]))

1 1 -1


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho list ứng viên.

### Cách 4 — Sắp xếp theo `(d, i)` rồi lấy phần tử đầu

Sắp xếp các ứng viên (`O(k log k)`) rồi lấy đầu. Không tối ưu hơn `min`, nhưng thể hiện cách "xếp hạng": hữu ích nếu sau này cần **top-k** drone gần nhất.

In [11]:
class SolutionSort:
    def nearestDrone(self, drones, target):
        tx, ty = target
        ranked = sorted((abs(x - tx) + abs(y - ty), i) for i, (x, y, r) in enumerate(drones)
                        if abs(x - tx) + abs(y - ty) <= r)
        return ranked[0][1] if ranked else -1

print(SolutionSort().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]),
      SolutionSort().nearestDrone([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]),
      SolutionSort().nearestDrone([[4, 4, 5]], [8, 6]))

1 1 -1


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 1 với `drones = [[2,1,5],[4,4,5],[6,6,8]]`, `target = (5,5)`:

| `i` | `(x, y, r)` | `d` | `d <= r`? | `d < best_d`? | `(best_d, best_i)` |
|---|---|---|---|---|---|
| (đầu) | | | | | `(inf, -1)` |
| 0 | `(2,1,5)` | 7 | không | - | `(inf, -1)` |
| 1 | `(4,4,5)` | 2 | có | `2 < inf` có | `(2, 1)` |
| 2 | `(6,6,8)` | 2 | có | `2 < 2` **không** | `(2, 1)` |

Đáp án `1` (giữ drone 1 nhờ `<` chặt).

In [12]:
def trace_loop(drones, target):
    tx, ty = target
    best_d, best_i = float("inf"), -1
    print("target =", target)
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - tx) + abs(y - ty)
        in_range = d <= r
        better = d < best_d
        if in_range and better:
            best_d, best_i = d, i
        print(f"  i={i}: d={d}, r={r}, tiếp cận được={in_range}, tốt hơn hẳn={better} -> best=({best_d}, {best_i})")
    return best_i

assert trace_loop([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]) == 1
assert trace_loop([[4, 4, 5]], [8, 6]) == -1
assert trace_loop([[0, 0, 8], [2, 2, 9]], [3, 4]) == 1

target = [5, 5]
  i=0: d=7, r=5, tiếp cận được=False, tốt hơn hẳn=True -> best=(inf, -1)
  i=1: d=2, r=5, tiếp cận được=True, tốt hơn hẳn=True -> best=(2, 1)
  i=2: d=2, r=8, tiếp cận được=True, tốt hơn hẳn=False -> best=(2, 1)
target = [8, 6]
  i=0: d=6, r=5, tiếp cận được=False, tốt hơn hẳn=True -> best=(inf, -1)
target = [3, 4]
  i=0: d=7, r=8, tiếp cận được=True, tốt hơn hẳn=True -> best=(7, 0)
  i=1: d=3, r=9, tiếp cận được=True, tốt hơn hẳn=True -> best=(3, 1)


In [13]:
def trace_tuple_min(drones, target):
    tx, ty = target
    cands = [(abs(x - tx) + abs(y - ty), i) for i, (x, y, r) in enumerate(drones) if abs(x - tx) + abs(y - ty) <= r]
    print("các ứng viên (d, i):", cands, "->", min(cands) if cands else None)
    return min(cands)[1] if cands else -1

assert trace_tuple_min([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]) == 1

các ứng viên (d, i): [(2, 1), (2, 2)] -> (2, 1)


## 9. Code nộp LeetCode

Dùng Cách 1 (vòng lặp, `O(1)` bộ nhớ phụ).

In [14]:
from typing import List, Optional


class Solution:
    def nearestDrone(self, drones: List[List[int]], target: List[int]) -> int:
        tx, ty = target
        best_d, best_i = float("inf"), -1
        for i, (x, y, r) in enumerate(drones):
            d = abs(x - tx) + abs(y - ty)
            if d <= r and d < best_d:
                best_d, best_i = d, i
        return best_i


print(Solution().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]),
      Solution().nearestDrone([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]),
      Solution().nearestDrone([[4, 4, 5]], [8, 6]))

1 1 -1


## 10. Vì sao thuật toán đúng

1. **Tập hợp hợp lệ.** `R = { i : d_i <= range_i }` là các drone tiếp cận được, với `d_i = |x_i - tx| + |y_i - ty|`. Đáp án phải thuộc `R`, hoặc là `-1` nếu `R` rỗng.
2. **Mục tiêu.** Chọn `i* in R` sao cho `(d_{i*}, i*)` nhỏ nhất theo thứ tự từ điển: khoảng cách nhỏ nhất, hòa thì chỉ số nhỏ nhất. Đó đúng là yêu cầu của đề.
3. **Invariant của vòng lặp.** Sau khi xét các drone `0..k-1`, `(best_d, best_i)` bằng khóa nhỏ nhất `(d, i)` trên `R ∩ {0..k-1}` (hoặc `(inf, -1)` nếu rỗng).
   - Khởi tạo: `k = 0`, tập rỗng, đúng.
   - Bước: xét `k`. Nếu `k` không thuộc `R` thì không đổi. Nếu `k` thuộc `R` và `d_k < best_d` thì `(d_k, k)` nhỏ hơn mọi khóa cũ, cập nhật. Nếu `d_k == best_d` thì khóa cũ `(best_d, best_i)` có `best_i < k` (vì ta duyệt tăng dần) nên **khóa cũ nhỏ hơn**, giữ nguyên (đó là lý do dùng `<` chặt). Nếu `d_k > best_d` thì không đổi.
4. **Kết luận.** Khi `k = n`, `best_i` là đáp án (hoặc `-1` nếu `R` rỗng).
5. Các cách `min`/`sorted` trên `(d, i)` tính cùng khóa nhỏ nhất theo thứ tự từ điển của tuple.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(drones) <= 100`.

- Cách 1 (nộp): Time `O(n)`, Space `O(1)`.
- Cách 2 (`min` + generator): Time `O(n)`, Space `O(1)`.
- Cách 3 (list ứng viên): Time `O(n)`, Space `O(n)`.
- Cách 4 (`sorted`): Time `O(n log n)`, Space `O(n)`.

Phải xét từng drone ít nhất một lần nên `O(n)` là tối ưu.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionLoop(), SolutionMin(), SolutionFilter(), SolutionSort(), Solution()]

fixed = [
    ([[0, 0, 8], [2, 2, 9]], [3, 4], 1),
    ([[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5], 1),
    ([[4, 4, 5]], [8, 6], -1),
    ([[0, 0, 1]], [0, 0], 0),                       # cùng vị trí: khoảng cách 0 <= tầm
    ([[0, 0, 2]], [1, 1], 0),                       # d = 2 == r: đúng bằng tầm vẫn được
    ([[0, 0, 1]], [1, 1], -1),                      # d = 2 > r
    ([[3, 3, 10], [3, 3, 10]], [0, 0], 0),          # hòa hoàn toàn: chỉ số nhỏ
    ([[-25, -25, 100], [25, 25, 1]], [25, 25], 1),
    ([[10, 10, 5], [0, 0, 20], [0, 1, 20]], [0, 0], 1),
]
for drones, target, expected in fixed:
    for sol in solutions:
        assert sol.nearestDrone([list(d) for d in drones], list(target)) == expected, (type(sol).__name__, drones, target)
print("test cố định OK:", len(fixed), "ca")

def ref(drones, target):
    best = None
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        if d <= r and (best is None or (d, i) < best):
            best = (d, i)
    return -1 if best is None else best[1]

random.seed(4024)
cnt = {"-1": 0, "ok": 0}
for _ in range(800):
    n = random.randint(1, 100)
    drones = [[random.randint(-25, 25), random.randint(-25, 25), random.randint(1, random.choice([5, 20, 100]))] for _ in range(n)]
    target = [random.randint(-25, 25), random.randint(-25, 25)]
    expected = ref(drones, target)
    cnt["-1" if expected == -1 else "ok"] += 1
    for sol in solutions:
        assert sol.nearestDrone([list(d) for d in drones], list(target)) == expected, (type(sol).__name__, drones, target)
print("test ngẫu nhiên OK, phân bố:", cnt)
assert cnt["-1"] > 10 and cnt["ok"] > 100

test cố định OK: 9 ca
test ngẫu nhiên OK, phân bố: {'-1': 25, 'ok': 775}


### Test thuộc tính (property test)

(1) Kết quả là `-1` hoặc chỉ số hợp lệ `0 <= i < n`, và drone đó **thực sự** tiếp cận được. (2) Hoán vị các drone: khoảng cách nhỏ nhất của kết quả không đổi. (3) Thêm một drone **không tiếp cận được** vào cuối không đổi kết quả. (4) Tăng mọi `range` thêm một lượng không làm kết quả trở về `-1`.

In [16]:
random.seed(40240)
for _ in range(400):
    n = random.randint(1, 30)
    drones = [[random.randint(-25, 25), random.randint(-25, 25), random.randint(1, 30)] for _ in range(n)]
    target = [random.randint(-25, 25), random.randint(-25, 25)]
    r = Solution().nearestDrone(drones, target)
    dist = lambda d: abs(d[0] - target[0]) + abs(d[1] - target[1])
    if r != -1:
        assert 0 <= r < n and dist(drones[r]) <= drones[r][2]
        sh = drones[:]; random.shuffle(sh)
        r2 = Solution().nearestDrone(sh, target)
        assert dist(sh[r2]) == dist(drones[r])
        bigger = [[x, y, rg + 5] for x, y, rg in drones]
        assert Solution().nearestDrone(bigger, target) != -1
    far = [target[0] + 40, target[1], 1]
    assert Solution().nearestDrone(drones + [far], target) == r
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên kiểm tra `range` (chọn drone gần nhất dù không tiếp cận được)

Ví dụ 3: drone duy nhất cách `6` nhưng tầm `5`; không thể tiếp cận -> `-1`, chứ không phải `0`.

In [17]:
def wrong_no_range(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        if d < best_d:                                             # SAI: thiếu d <= r
            best_d, best_i = d, i
    return best_i

print(wrong_no_range([[4, 4, 5]], [8, 6]), "!=", Solution().nearestDrone([[4, 4, 5]], [8, 6]))
assert wrong_no_range([[4, 4, 5]], [8, 6]) == 0 and Solution().nearestDrone([[4, 4, 5]], [8, 6]) == -1

0 != -1


### Lỗi 2: dùng `d < r` (chặt) thay `d <= r`

Đề cho phép khoảng cách **đúng bằng** tầm bay.

In [18]:
def wrong_strict_range(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        if d < r and d < best_d:                                   # SAI: phải d <= r
            best_d, best_i = d, i
    return best_i

print(wrong_strict_range([[0, 0, 2]], [1, 1]), "!=", Solution().nearestDrone([[0, 0, 2]], [1, 1]))
assert wrong_strict_range([[0, 0, 2]], [1, 1]) == -1 and Solution().nearestDrone([[0, 0, 2]], [1, 1]) == 0

-1 != 0


### Lỗi 3: dùng `<=` khi cập nhật (sai tie-break)

Với `d <= best_d`, khi hòa ta đổi sang drone **sau** (chỉ số lớn hơn). Đề yêu cầu chỉ số **nhỏ nhất**.

In [19]:
def wrong_tie(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        if d <= r and d <= best_d:                                 # SAI: <= làm lấy chỉ số lớn khi hòa
            best_d, best_i = d, i
    return best_i

drones, target = [[2, 1, 5], [4, 4, 5], [6, 6, 8]], [5, 5]
print(wrong_tie(drones, target), "!=", Solution().nearestDrone(drones, target))
assert wrong_tie(drones, target) == 2 and Solution().nearestDrone(drones, target) == 1

2 != 1


### Lỗi 4: dùng khoảng cách Euclid thay vì Manhattan

Đề dùng Manhattan. Euclid cho thứ tự khác: drone `(2, 2)` và `(5, 1)`... ví dụ drone A cách target `(3,3)` Manhattan 6 (Euclid 4.24) còn drone B cách `(0,5)` Manhattan 5 (Euclid 5): Manhattan chọn B, Euclid chọn A.

In [20]:
def wrong_euclid(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = ((x - target[0]) ** 2 + (y - target[1]) ** 2) ** 0.5     # SAI: không phải Manhattan
        if abs(x - target[0]) + abs(y - target[1]) <= r and d < best_d:
            best_d, best_i = d, i
    return best_i

drones, target = [[3, 3, 10], [0, 5, 10]], [0, 0]
print(wrong_euclid(drones, target), "!=", Solution().nearestDrone(drones, target))
assert wrong_euclid(drones, target) == 0 and Solution().nearestDrone(drones, target) == 1

0 != 1


### Lỗi 5: trả về khoảng cách thay vì chỉ số

Đề yêu cầu **chỉ số** của drone, không phải khoảng cách của nó.

In [21]:
def wrong_return_distance(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = abs(x - target[0]) + abs(y - target[1])
        if d <= r and d < best_d:
            best_d, best_i = d, i
    return best_d if best_i != -1 else -1                          # SAI: trả khoảng cách

print(wrong_return_distance([[0, 0, 8], [2, 2, 9]], [3, 4]), "!=", Solution().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]))
assert wrong_return_distance([[0, 0, 8], [2, 2, 9]], [3, 4]) == 3 and Solution().nearestDrone([[0, 0, 8], [2, 2, 9]], [3, 4]) == 1

3 != 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn **nhỏ nhất**, hòa thì **chỉ số nhỏ nhất**" -> duyệt theo chỉ số tăng và chỉ cập nhật khi `<` **chặt**, hoặc dùng `min` trên tuple `(giá_trị, chỉ_số)`.
- "Lọc điều kiện **rồi** chọn" -> viết điều kiện lọc ở một chỗ (`if d <= r`) để không quên.
- Manhattan = `abs(dx) + abs(dy)`; Euclid không dùng khi đề nói "Manhattan".
- `min(generator, default=...)` tránh lỗi khi không có ứng viên.
- Với mảng 2 chiều `drones[i] = [x, y, r]`, giải nén `for i, (x, y, r) in enumerate(drones)` cho code rõ ràng.
- Thử các ca: không ai tiếp cận được, hòa hoàn toàn, khoảng cách **đúng bằng** tầm, hai drone cùng vị trí.

## 15. Biến thể của bài

1. **Find Nearest Point That Has the Same X or Y Coordinate** (1779): chọn điểm gần nhất theo Manhattan, hòa thì chỉ số nhỏ.
2. **Minimum Time Visiting All Points** (1266), **K Closest Points to Origin** (973): khoảng cách và top-k.
3. **Special Positions in a Binary Matrix** (1582): lọc điều kiện trên lưới.
4. Biến thể: trả về **k drone gần nhất** (dùng `heapq.nsmallest` hoặc sắp xếp).
5. Biến thể: nhiều mục tiêu: với mỗi mục tiêu tìm drone gần nhất (`O(n m)`).
6. Biến thể: tầm bay tính theo **Euclid** hay theo **Chebyshev** (`max(|dx|, |dy|)`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `<` chặt | theo dõi `best_d`, `best_i` | `O(n)` | `O(1)` | **Nộp bài** |
| 2. `min` trên `(d, i)` | tie-break nằm trong khóa | `O(n)` | `O(1)` | Gọn, ít lỗi tie-break |
| 3. Lọc ra list | in được các ứng viên | `O(n)` | `O(n)` | Gỡ lỗi |
| 4. `sorted` | xếp hạng toàn bộ | `O(n log n)` | `O(n)` | Khi cần top-k |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().nearestDrone([[0, 0, 3], [1, 1, 3]], [2, 2])` và `Solution().nearestDrone([[5, 5, 1]], [5, 6])` ra gì?

**Bài 2 (tính tay):** `drones = [[1,1,4],[3,3,2],[0,5,6]]`, `target = [2,3]`: tính khoảng cách mỗi drone, drone nào tiếp cận được, đáp án.

**Bài 3 (điền TODO):** hoàn thành `k_nearest(drones, target, k)`: danh sách chỉ số của `k` drone tiếp cận được gần nhất (hòa -> chỉ số nhỏ).

**Bài 4 (tự viết lại):** viết `count_reachable(drones, target)`: số drone tiếp cận được.

**Bài 5 (mini challenge):** viết `nearest_chebyshev(drones, target)`: dùng khoảng cách Chebyshev `max(|dx|, |dy|)` thay Manhattan; tìm trường hợp mà đáp án khác bài gốc.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().nearestDrone([[0, 0, 3], [1, 1, 3]], [2, 2]), Solution().nearestDrone([[5, 5, 1]], [5, 6]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def k_nearest_todo(drones, target, k):
    # TODO: lọc ứng viên (d, i), sắp xếp, lấy k chỉ số đầu
    return []

1 0


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: tính Manhattan của từng drone tới `(2, 2)`; drone `(5,5,1)` tới `(5,6)` cách `1`. Bài 2: `d` của ba drone.

**Mức 2 (vừa):** Bài 3: `sorted(...)[:k]` rồi lấy chỉ số. Bài 4: `sum(1 for ... if d <= r)`. Bài 5: thay `abs(dx) + abs(dy)` bằng `max(abs(dx), abs(dy))`.

**Mức 3 (gần đáp án):** Bài 1: `1` (drone 0 cách `4 > 3` loại; drone 1 cách `2`) và `0` (`1 <= 1`). Bài 2: khoảng cách `3, 1, 4`; tiếp cận được: drone 0 (`3 <= 4`), drone 1 (`1 <= 2`), drone 2 (`4 <= 6`) -> đáp án `1`. Bài 3: `k_nearest(..., 2) = [1, 0]`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().nearestDrone([[0, 0, 3], [1, 1, 3]], [2, 2]) == 1 and Solution().nearestDrone([[5, 5, 1]], [5, 6]) == 0
d2 = [[1, 1, 4], [3, 3, 2], [0, 5, 6]]
assert [abs(x - 2) + abs(y - 3) for x, y, r in d2] == [3, 1, 4] and Solution().nearestDrone(d2, [2, 3]) == 1

# Bài 3
def k_nearest(drones, target, k):
    ranked = sorted((abs(x - target[0]) + abs(y - target[1]), i) for i, (x, y, r) in enumerate(drones)
                    if abs(x - target[0]) + abs(y - target[1]) <= r)
    return [i for _, i in ranked[:k]]

assert k_nearest(d2, [2, 3], 2) == [1, 0] and k_nearest(d2, [2, 3], 10) == [1, 0, 2] and k_nearest([[4, 4, 5]], [8, 6], 1) == []

# Bài 4
def count_reachable(drones, target):
    return sum(1 for x, y, r in drones if abs(x - target[0]) + abs(y - target[1]) <= r)

assert count_reachable(d2, [2, 3]) == 3 and count_reachable([[4, 4, 5]], [8, 6]) == 0

# Bài 5
def nearest_chebyshev(drones, target):
    best_d, best_i = float("inf"), -1
    for i, (x, y, r) in enumerate(drones):
        d = max(abs(x - target[0]), abs(y - target[1]))
        if d <= r and d < best_d:
            best_d, best_i = d, i
    return best_i

diff = [([[3, 3, 10], [0, 5, 10]], [0, 0])]
for dr, tg in diff:
    assert nearest_chebyshev(dr, tg) == 0 and Solution().nearestDrone(dr, tg) == 1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def nearestDrone(self, drones: List[List[int]], target: List[int]) -> int:
        tx, ty = target
        best_d, best_i = float("inf"), -1
        for i, (x, y, r) in enumerate(drones):
            d = abs(x - tx) + abs(y - ty)          # khoảng cách Manhattan
            if d <= r and d < best_d:               # tiếp cận được VÀ tốt hơn hẳn (giữ chỉ số nhỏ khi hòa)
                best_d, best_i = d, i
        return best_i
```

Khung chung "lọc điều kiện rồi chọn nhỏ nhất, hòa thì chỉ số nhỏ": cập nhật bằng `<` chặt hoặc `min` trên `(giá_trị, chỉ_số)`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Có điều kiện `d <= r` (không chặt), dùng Manhattan, cập nhật bằng `d < best_d` (chặt), trả về **chỉ số**.
- [ ] Đã thử: không drone nào tiếp cận được (`-1`), hòa khoảng cách, `d == r`.